# Base model inference: composed-transformation misconception diagnoser

Day 1 setup and litmus baseline for the "Train Your Own Small Learning Model" project.

This notebook gets the base model running on Colab and measures how it does on the target behavior *before* any fine-tuning, so there is a base baseline to beat.

**Base model:** `unsloth/Qwen3-4B-Instruct-2507-bnb-4bit`. The Instruct-2507 variant is non-thinking, so it can emit clean JSON with no chain-of-thought prose (the spec forbids prose outside the JSON). Switch `MODEL_NAME` to `unsloth/Qwen3-1.7B` if generation is slow or memory is tight.

**Runtime:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all.

**Behavior spec (from the Brainlift):** given a two-step composed rigid-motion problem and a student's answer, always return one valid JSON object with fields `target_quantity`, `correct_answer`, `misconception_label`, `evidence_span`, `hint`, and nothing outside it; label the misconception from a fixed taxonomy; the hint never states the correct final answer.

In [ ]:
# Confirm a GPU is attached (expect a T4 on free Colab).
!nvidia-smi

In [ ]:
# Install Unsloth (also sets up the QLoRA stack for training later).
# Output is NOT captured, so any dependency error is visible. Takes about 1-2 min.
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo
# If pip upgrades torch, do Runtime > Restart session once, then run the cells below (skip this one).

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507-bnb-4bit"  # swap to "unsloth/Qwen3-1.7B" if slow or OOM
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)  # enable Unsloth's faster inference path
print("loaded:", MODEL_NAME)

In [ ]:
# v0 misconception taxonomy: a DRAFT starter set to validate later, and the error types we will
# inject when generating data. The Brainlift's finding is that no validated taxonomy exists yet,
# so treat these labels as provisional.
MISCONCEPTIONS = {
    "order_swapped": "Applied the two transformations in the wrong order.",
    "center_not_updated": "Did not update the center or reference frame for the second transformation.",
    "wrong_direction": "Rotated or translated in the wrong direction or sign.",
    "reflected_wrong_axis": "Reflected over the wrong axis.",
    "only_first_applied": "Applied only the first transformation and skipped the second.",
    "arithmetic_slip": "Correct method but a small coordinate arithmetic error.",
    "none": "The student's answer is correct.",
}

SYSTEM_PROMPT = (
    "You diagnose a student's error on a two-step composed geometric transformation.\n"
    "Return a SINGLE valid JSON object and nothing else: no prose, no markdown, no code fences.\n"
    "Required fields:\n"
    '  "target_quantity": what is being asked for (for example, the image coordinates of the vertices).\n'
    '  "correct_answer": the correct result, which you compute yourself.\n'
    f'  "misconception_label": exactly one of {sorted(MISCONCEPTIONS)}.\n'
    '  "evidence_span": the part of the student\'s answer that shows the error.\n'
    '  "hint": a Socratic nudge that does NOT state the correct final answer.\n'
    'Use "none" only when the student is correct. The hint must never reveal correct_answer.'
)
print(SYSTEM_PROMPT)

In [ ]:
# Exact integer rigid motions, so every problem carries programmatic ground truth
# (no frontier teacher needed to compute the correct answer).
def translate(pts, dx, dy):
    return [(x + dx, y + dy) for (x, y) in pts]

def rotate(pts, center, deg):
    cx, cy = center
    out = []
    for (x, y) in pts:
        x0, y0 = x - cx, y - cy
        if deg == 90:
            xr, yr = -y0, x0
        elif deg == 180:
            xr, yr = -x0, -y0
        elif deg == 270:
            xr, yr = y0, -x0
        else:
            raise ValueError(f"unsupported angle: {deg}")
        out.append((xr + cx, yr + cy))
    return out

def reflect(pts, axis):
    fns = {
        "x": lambda x, y: (x, -y),
        "y": lambda x, y: (-x, y),
        "y=x": lambda x, y: (y, x),
        "y=-x": lambda x, y: (-y, -x),
    }
    f = fns[axis]
    return [f(x, y) for (x, y) in pts]

def fmt(pts):
    return ", ".join(f"({x}, {y})" for (x, y) in pts)

In [ ]:
# Build a couple of two-step problems and inject a specific misconception into the student's answer.
def problem_text(pre, step1_desc, step2_desc):
    return (
        f"Triangle with vertices {fmt(pre)} undergoes two transformations:\n"
        f"  Step 1: {step1_desc}\n"
        f"  Step 2: {step2_desc}\n"
        "Give the vertices of the final image."
    )

PRE = [(1, 1), (4, 1), (1, 3)]
S1_DESC = "translate by (2, -1)"
S2_DESC = "rotate 90 degrees counterclockwise about the origin"

step1 = lambda p: translate(p, 2, -1)
step2 = lambda p: rotate(p, (0, 0), 90)

correct = step2(step1(PRE))          # translate THEN rotate (the correct order)
ans_order_swapped = step1(step2(PRE))  # rotate THEN translate
ans_only_first = step1(PRE)            # forgot step 2

samples = [
    {"text": problem_text(PRE, S1_DESC, S2_DESC), "correct": correct,
     "student": ans_order_swapped, "true_label": "order_swapped"},
    {"text": problem_text(PRE, S1_DESC, S2_DESC), "correct": correct,
     "student": ans_only_first, "true_label": "only_first_applied"},
]

for s in samples:
    print(f"{s['true_label']:>20} | correct: {fmt(s['correct'])} | student: {fmt(s['student'])}")

In [ ]:
import json, re

def diagnose(problem, student):
    user = f"{problem}\n\nStudent's answer: {fmt(student)}"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    output = model.generate(input_ids=input_ids, max_new_tokens=512, do_sample=False)
    text = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)
    return text.strip()

def check(text, sample):
    """Preview of the eval: the same programmatic checks the harness will use."""
    result = {"valid_json": False, "json_only": False, "label_valid": False,
              "label_correct": None, "no_leak": None}
    obj = None
    try:
        obj = json.loads(text)
        result["valid_json"] = True
        result["json_only"] = True
    except Exception:
        m = re.search(r"\{.*\}", text, re.S)
        if m:
            try:
                obj = json.loads(m.group(0))
                result["valid_json"] = True  # parseable, but wrapped in extra text
            except Exception:
                obj = None
    if isinstance(obj, dict):
        label = obj.get("misconception_label")
        result["label_valid"] = label in MISCONCEPTIONS
        result["label_correct"] = (label == sample["true_label"])
        hint = str(obj.get("hint", ""))
        result["no_leak"] = all(
            f"({x}, {y})" not in hint and f"({x},{y})" not in hint
            for (x, y) in sample["correct"]
        )
    return result

In [ ]:
# Run the base model on the samples. This is the baseline the fine-tune has to beat.
for i, s in enumerate(samples, 1):
    print(f"\n===== sample {i}: expected label = {s['true_label']} =====")
    raw = diagnose(s["text"], s["student"])
    print(raw)
    print("check:", check(raw, s))

## Reading the baseline

For each sample, look at:
- `json_only` / `valid_json`: did it return one clean JSON object with no extra prose?
- `label_correct`: did it name the actual injected misconception?
- `no_leak`: did the hint avoid stating the correct coordinates?

Expect the base model to slip on at least one of these, usually the composition itself (it computes the wrong `correct_answer`) or the JSON discipline. That gap is what fine-tuning has to close, and it is the base column of your base-vs-tuned table.

## Next steps
1. Turn the transform and injection helpers into a full data generator: hundreds to a few thousand labeled examples, balanced across the taxonomy, with varied polygons, transformations, centers, and axes.
2. Build the eval harness: an LLM-as-judge on the spec plus these programmatic checks (valid JSON, no leak, correct label), run base vs tuned vs a prompted frontier model on a held-out set.
3. QLoRA fine-tune with Unsloth, then re-run the eval and report the deltas.